In [ ]:
# Install Required Libraries
!pip -q install transformers
!pip -q install timm
!pip -q install accelerate

# Imports
import os
import random
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from transformers import SegformerForSemanticSegmentation

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device : {device}")

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Dataset Paths

X_PATH = "/content/drive/MyDrive/NEW_ROI_X.npy"
Y_PATH = "/content/drive/MyDrive/NEW_ROI_Y.npy"

# Load Arrays

X = np.load(X_PATH)
Y = np.load(Y_PATH)

print("Images Shape :", X.shape)
print("Masks Shape  :", Y.shape)

In [ ]:
print("Image dtype :", X.dtype)
print("Mask dtype  :", Y.dtype)

print("Image Min :", X.min())
print("Image Max :", X.max())

print("Mask Unique Values :", np.unique(Y))

In [ ]:
idx = np.random.randint(len(X))

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(X[idx])
plt.title("Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(Y[idx], cmap="gray")
plt.title("Mask")
plt.axis("off")

plt.show()

In [ ]:
# Train / Validation / Test Split

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.50,
    random_state=42
)

print(f"Train : {len(X_train)}")
print(f"Validation : {len(X_val)}")
print(f"Test : {len(X_test)}")

In [ ]:
class BreastDataset(Dataset):

    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image = self.images[idx]
        mask = self.masks[idx]

        # Convert grayscale (128,128,1) → RGB (128,128,3)
        image = np.repeat(image, 3, axis=-1)

        image = torch.tensor(image, dtype=torch.float32)
        image = image.permute(2, 0, 1)

        mask = torch.tensor(mask, dtype=torch.float32)
        mask = mask.squeeze(2)

        return image, mask

In [ ]:
train_dataset = BreastDataset(X_train, Y_train)
val_dataset = BreastDataset(X_val, Y_val)
test_dataset = BreastDataset(X_test, Y_test)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
images, masks = next(iter(train_loader))

print("Images :", images.shape)
print("Masks :", masks.shape)

In [ ]:
image = images[0].permute(1, 2, 0).numpy()
mask = masks[0].numpy()

plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.imshow(image[:, :, 0], cmap='gray')
plt.title("Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap='gray')
plt.title("Mask")
plt.axis("off")

plt.show()

In [ ]:
from transformers import SegformerForSemanticSegmentation

# Load Pretrained SegFormer-B0

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=1,
    ignore_mismatched_sizes=True
)

model.to(device)

print("Model loaded successfully!")

In [ ]:
print(model)

In [ ]:
model.eval()

images, masks = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    outputs = model(pixel_values=images)

print("Output shape :", outputs.logits.shape)

In [ ]:
logits = outputs.logits

print("Minimum :", logits.min().item())
print("Maximum :", logits.max().item())

In [ ]:
prediction = torch.sigmoid(logits[0]).cpu().numpy().squeeze()

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(images[0].cpu().permute(1,2,0)[:,:,0], cmap='gray')
plt.title("Input")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(masks[0], cmap='gray')
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(prediction, cmap='gray')
plt.title("Raw Prediction")
plt.axis("off")

plt.show()

In [ ]:
# Part 4

import torch.nn.functional as F

class DiceLoss(nn.Module):

    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.sigmoid(logits)

        probs = probs.view(-1)
        targets = targets.view(-1)

        intersection = (probs * targets).sum()

        dice = (2. * intersection + self.smooth) / (
            probs.sum() + targets.sum() + self.smooth
        )

        return 1 - dice

In [ ]:
class BCEDiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):

        bce = self.bce(logits, targets)

        dice = self.dice(logits, targets)

        return bce + dice

In [ ]:
criterion = BCEDiceLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

model.eval()

with torch.no_grad():

    outputs = model(pixel_values=images)

    logits = outputs.logits

    logits = F.interpolate(
        logits,
        size=masks.shape[-2:],      # (128,128)
        mode="bilinear",
        align_corners=False
    )

loss = criterion(logits, masks.unsqueeze(1))

print("Logits :", logits.shape)
print("Masks  :", masks.unsqueeze(1).shape)
print("Loss   :", loss.item())

In [ ]:
import torch.nn.functional as F

def calculate_metrics(logits, masks, threshold=0.5):

    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds = preds.view(-1)
    masks = masks.view(-1)

    TP = (preds * masks).sum()
    FP = (preds * (1 - masks)).sum()
    FN = ((1 - preds) * masks).sum()

    dice = (2 * TP + 1e-6) / (2 * TP + FP + FN + 1e-6)
    iou = (TP + 1e-6) / (TP + FP + FN + 1e-6)
    precision = (TP + 1e-6) / (TP + FP + 1e-6)
    recall = (TP + 1e-6) / (TP + FN + 1e-6)
    f1 = (2 * precision * recall) / (precision + recall + 1e-6)

    return (
        dice.item(),
        iou.item(),
        precision.item(),
        recall.item(),
        f1.item()
    )

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0

    dice_total = 0
    iou_total = 0
    precision_total = 0
    recall_total = 0
    f1_total = 0

    for images, masks in tqdm(loader):

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values=images)

        logits = outputs.logits

        logits = F.interpolate(
            logits,
            size=masks.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        loss = criterion(logits, masks.unsqueeze(1))

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        dice, iou, precision, recall, f1 = calculate_metrics(
            logits,
            masks.unsqueeze(1)
        )

        dice_total += dice
        iou_total += iou
        precision_total += precision
        recall_total += recall
        f1_total += f1

    n = len(loader)

    return {
        "loss": running_loss / n,
        "dice": dice_total / n,
        "iou": iou_total / n,
        "precision": precision_total / n,
        "recall": recall_total / n,
        "f1": f1_total / n
    }

In [ ]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0

    dice_total = 0
    iou_total = 0
    precision_total = 0
    recall_total = 0
    f1_total = 0

    with torch.no_grad():

        for images, masks in tqdm(loader):

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(pixel_values=images)

            logits = outputs.logits

            logits = F.interpolate(
                logits,
                size=masks.shape[-2:],
                mode='bilinear',
                align_corners=False
            )

            loss = criterion(logits, masks.unsqueeze(1))

            running_loss += loss.item()

            dice, iou, precision, recall, f1 = calculate_metrics(
                logits,
                masks.unsqueeze(1)
            )

            dice_total += dice
            iou_total += iou
            precision_total += precision
            recall_total += recall
            f1_total += f1

    n = len(loader)

    return {
        "loss": running_loss / n,
        "dice": dice_total / n,
        "iou": iou_total / n,
        "precision": precision_total / n,
        "recall": recall_total / n,
        "f1": f1_total / n
    }

In [ ]:
NUM_EPOCHS = 30

best_val_loss = float("inf")

patience = 7
counter = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_dice": [],
    "val_dice": [],
    "train_iou": [],
    "val_iou": []
}

In [ ]:
for epoch in range(NUM_EPOCHS):

    print("=" * 70)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print("=" * 70)

    train_metrics = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_metrics = validate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_metrics["loss"])

    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])

    history["train_dice"].append(train_metrics["dice"])
    history["val_dice"].append(val_metrics["dice"])

    history["train_iou"].append(train_metrics["iou"])
    history["val_iou"].append(val_metrics["iou"])

    print(f"Train Loss : {train_metrics['loss']:.4f}")
    print(f"Train Dice : {train_metrics['dice']:.4f}")
    print(f"Train IoU  : {train_metrics['iou']:.4f}")

    print()

    print(f"Val Loss   : {val_metrics['loss']:.4f}")
    print(f"Val Dice   : {val_metrics['dice']:.4f}")
    print(f"Val IoU    : {val_metrics['iou']:.4f}")

    if val_metrics["loss"] < best_val_loss:

        best_val_loss = val_metrics["loss"]

        torch.save(model.state_dict(), "best_segformer.pth")

        counter = 0

        print("\n✅ Best model saved.")

    else:

        counter += 1

        print(f"\nEarly Stopping Counter : {counter}/{patience}")

        if counter >= patience:

            print("\nTraining stopped.")
            break

In [ ]:
# Load Best Model

model.load_state_dict(torch.load("best_segformer.pth", map_location=device))
model.to(device)
model.eval()

print("Best model loaded successfully!")

In [ ]:
def evaluate_model(model, loader, criterion, device):

    model.eval()

    running_loss = 0

    dice_total = 0
    iou_total = 0
    precision_total = 0
    recall_total = 0
    f1_total = 0

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(pixel_values=images)

            logits = outputs.logits

            # Upsample logits to original mask size
            logits = F.interpolate(
                logits,
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            loss = criterion(logits, masks.unsqueeze(1))

            running_loss += loss.item()

            dice, iou, precision, recall, f1 = calculate_metrics(
                logits,
                masks.unsqueeze(1)
            )

            dice_total += dice
            iou_total += iou
            precision_total += precision
            recall_total += recall
            f1_total += f1

    n = len(loader)

    return {
        "Loss": running_loss / n,
        "Dice": dice_total / n,
        "IoU": iou_total / n,
        "Precision": precision_total / n,
        "Recall": recall_total / n,
        "F1": f1_total / n
    }

In [ ]:
train_results = evaluate_model(model, train_loader, criterion, device)
val_results   = evaluate_model(model, val_loader, criterion, device)
test_results  = evaluate_model(model, test_loader, criterion, device)

In [ ]:
from tabulate import tabulate

table = [
    [
        "Train",
        train_results["Loss"],
        train_results["Dice"],
        train_results["IoU"],
        train_results["Precision"],
        train_results["Recall"],
        train_results["F1"]
    ],
    [
        "Validation",
        val_results["Loss"],
        val_results["Dice"],
        val_results["IoU"],
        val_results["Precision"],
        val_results["Recall"],
        val_results["F1"]
    ],
    [
        "Test",
        test_results["Loss"],
        test_results["Dice"],
        test_results["IoU"],
        test_results["Precision"],
        test_results["Recall"],
        test_results["F1"]
    ]
]

headers = [
    "Dataset",
    "Loss",
    "Dice",
    "IoU",
    "Precision",
    "Recall",
    "F1"
]

print(tabulate(table, headers=headers, floatfmt=".4f", tablefmt="grid"))

In [ ]:
model.eval()

images, masks = next(iter(test_loader))

images = images.to(device)

with torch.no_grad():

    outputs = model(pixel_values=images)

    logits = outputs.logits

    logits = F.interpolate(
        logits,
        size=masks.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

    predictions = torch.sigmoid(logits)
    predictions = (predictions > 0.5).float()

In [ ]:
num_samples = 5

plt.figure(figsize=(15, num_samples * 4))

for i in range(num_samples):

    # Input Image
    plt.subplot(num_samples, 3, i * 3 + 1)
    plt.imshow(images[i].cpu().permute(1, 2, 0)[:, :, 0], cmap='gray')
    plt.title("Input")
    plt.axis("off")

    # Ground Truth
    plt.subplot(num_samples, 3, i * 3 + 2)
    plt.imshow(masks[i], cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    # Prediction
    plt.subplot(num_samples, 3, i * 3 + 3)
    plt.imshow(predictions[i, 0].cpu(), cmap='gray')
    plt.title("Prediction")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history["train_dice"], label="Train Dice")
plt.plot(history["val_dice"], label="Validation Dice")

plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Training vs Validation Dice")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Save Model

torch.save(model.state_dict(), "segformer_final.pth")

print("Model saved successfully!")

In [ ]:
idx = random.randint(0, len(test_dataset)-1)

image, mask = test_dataset[idx]

image_tensor = image.unsqueeze(0).to(device)

with torch.no_grad():

    logits = model(pixel_values=image_tensor).logits

    logits = F.interpolate(
        logits,
        size=(128,128),
        mode="bilinear",
        align_corners=False
    )

    pred = (torch.sigmoid(logits)>0.5).float()

image = image.numpy()[0]
mask = mask.numpy()
pred = pred.squeeze().cpu().numpy()

plt.figure(figsize=(15,5))

plt.subplot(131)
plt.imshow(image,cmap='gray')
plt.title("Original")

plt.subplot(132)
plt.imshow(image,cmap='gray')
plt.imshow(mask,alpha=0.45,cmap='Reds')
plt.title("Ground Truth Overlay")

plt.subplot(133)
plt.imshow(image,cmap='gray')
plt.imshow(pred,alpha=0.45,cmap='Blues')
plt.title("Prediction Overlay")

plt.show()

In [ ]:
idx = random.randint(0, len(test_dataset)-1)

image, mask = test_dataset[idx]

image_tensor = image.unsqueeze(0).to(device)

with torch.no_grad():

    logits = model(pixel_values=image_tensor).logits

    logits = F.interpolate(
        logits,
        size=(128,128),
        mode="bilinear",
        align_corners=False
    )

    prob = torch.sigmoid(logits)

plt.figure(figsize=(10,4))

plt.subplot(121)
plt.imshow(image.numpy()[0], cmap='gray')
plt.title("Input")
plt.axis("off")

plt.subplot(122)
plt.imshow(prob.squeeze().cpu(), cmap='jet')
plt.colorbar()
plt.title("Probability Map")
plt.axis("off")

plt.show()